# BirdCLEF+ 2026 - exp002 Submission

**CPU Notebook / 推論専用 / 90分以内**

### Kaggle Notebook の Input に追加するもの
- `birdclef-2026` … コンペデータ
- `birdclef2026-exp002-weights` … Colabで学習した重み (best_fold0.pth)

### exp001 からの改善点
- **SED + AttBlockV2** モデル（アーキテクチャ一致必須）
- **重複推論**: 10秒ウィンドウを2.5秒ステップでスライド → 各5秒セグメントを複数チャンクで推論し平均
- **sigmoid推論**: CrossEntropyLoss で訓練 → sigmoid でクラス確率化


In [ ]:
!pip install -q timm

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import timm
from tqdm.notebook import tqdm

# Kaggle CPUノートブックは1分制限のためCPU固定
DEVICE = torch.device('cpu')
print(f'torch     : {torch.__version__}')
print(f'torchaudio: {torchaudio.__version__}')
print(f'device    : {DEVICE}')

In [ ]:
# ── パス自動検索 ──────────────────────────────────────────────
_w = glob.glob('/kaggle/input/**/best_fold0.pth', recursive=True)
WEIGHT_PATH = _w[0] if _w else '/kaggle/input/birdclef2026-exp002-weights/best_fold0.pth'

_s = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
COMP_DIR = os.path.dirname(_s[0]) if _s else '/kaggle/input/birdclef-2026'

SAMPLE_SUB_CSV = f'{COMP_DIR}/sample_submission.csv'
TEST_SOUND_DIR = f'{COMP_DIR}/test_soundscapes'

print(f'COMP_DIR     : {COMP_DIR}')
print(f'WEIGHT_PATH  : {WEIGHT_PATH}')
print(f'weight exists: {os.path.exists(WEIGHT_PATH)}')

In [ ]:
# ── ハイパーパラメータ（train時と同一にすること）─────────────
CFG = dict(
    sample_rate      = 32000,
    n_samples        = 32000 * 10,   # 10秒ウィンドウ
    n_mels           = 128,
    n_fft            = 1024,
    hop_length       = 320,
    fmin             = 20,
    fmax             = 16000,
    model_name       = 'tf_efficientnet_b0_ns',
    num_classes      = 234,
    infer_batch_size = 16,
    # 重複推論: 各5秒セグメントを複数の10秒ウィンドウで推論
    # ウィンドウの中心を end_sec ± offset で複数取る
    overlap_offsets  = [-2.5, 0.0, 2.5],  # 秒単位のオフセット
)

In [ ]:
# ── torchaudio による Mel 変換 ────────────────────────────────
mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate = CFG['sample_rate'],
        n_fft       = CFG['n_fft'],
        hop_length  = CFG['hop_length'],
        n_mels      = CFG['n_mels'],
        f_min       = CFG['fmin'],
        f_max       = CFG['fmax'],
    ),
    T.AmplitudeToDB(top_db=80),
)

def audio_to_melspec(audio_tensor: torch.Tensor) -> torch.Tensor:
    """(batch, n_samples) -> (batch, 1, n_mels, time)"""
    with torch.no_grad():
        mel = mel_transform(audio_tensor)             # (B, n_mels, T)
    mel = mel - mel.amin(dim=(-2, -1), keepdim=True)
    mel = mel / (mel.amax(dim=(-2, -1), keepdim=True) + 1e-8)
    return mel.unsqueeze(1)                           # (B, 1, n_mels, T)

In [ ]:
# ── モデル定義（train時と同一）────────────────────────────────
class AttBlockV2(nn.Module):
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.att = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        att = torch.softmax(torch.tanh(self.att(x)), dim=-1)
        cla = self.cla(x)
        return (att * cla).sum(dim=-1)


class BirdCLEFSED(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'], pretrained=False,
            in_chans=1, num_classes=0, global_pool='',
        )
        in_features    = self.backbone.num_features
        self.bn        = nn.BatchNorm1d(in_features)
        self.dropout   = nn.Dropout(p=0.3)
        self.att_block = AttBlockV2(in_features, CFG['num_classes'])

    def forward(self, x):
        feat = self.backbone.forward_features(x)  # (B, C, H, W)
        feat = feat.mean(dim=2)                   # freq pool → (B, C, W)
        feat = self.bn(feat)
        feat = self.dropout(feat)
        return self.att_block(feat)               # (B, num_classes)

In [ ]:
# ── モデルロード ──────────────────────────────────────────────
checkpoint = torch.load(WEIGHT_PATH, map_location='cpu')
model = BirdCLEFSED()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
LABELS = checkpoint['labels']
print(f'Loaded: epoch={checkpoint["epoch"]}, CV AUC={checkpoint["best_auc"]:.4f}')
print(f'Classes: {len(LABELS)}')

In [ ]:
# ── チャンク切り出し（重複推論対応）──────────────────────────
def extract_chunk(audio: torch.Tensor, center_sec: float) -> torch.Tensor:
    """center_sec を中心に n_samples 分を切り出す（端は0パディング）"""
    n, sr = CFG['n_samples'], CFG['sample_rate']
    center = int(center_sec * sr)
    start  = center - n // 2
    end    = start + n

    if start < 0:
        chunk = torch.cat([torch.zeros(-start), audio[:end]])
    elif end > len(audio):
        pad   = end - len(audio)
        chunk = torch.cat([audio[start:], torch.zeros(pad)])
    else:
        chunk = audio[start:end]

    # 長さ保証
    if len(chunk) < n:
        chunk = torch.cat([chunk, torch.zeros(n - len(chunk))])
    return chunk[:n]


def extract_chunks_overlap(audio: torch.Tensor, end_secs: list) -> torch.Tensor:
    """各 end_sec に対して複数オフセットのチャンクを生成する。
    返すテンソルの shape: (N * n_offsets, n_samples)
    """
    offsets = CFG['overlap_offsets']
    chunks  = []
    for end_sec in end_secs:
        # 5秒セグメントの中心 = end_sec - 2.5
        center_base = end_sec - 2.5
        for offset in offsets:
            center = center_base + offset
            chunks.append(extract_chunk(audio, center))
    return torch.stack(chunks)  # (N*n_offsets, n_samples)

In [ ]:
# ── 推論メインループ ──────────────────────────────────────────
sub_df = pd.read_csv(SAMPLE_SUB_CSV)

def parse_row_id(row_id):
    parts = row_id.rsplit('_', 1)
    return parts[0], int(parts[1])

sub_df[['stem', 'end_sec']] = sub_df['row_id'].apply(
    lambda x: pd.Series(parse_row_id(x))
)
all_preds = np.zeros((len(sub_df), len(LABELS)), dtype=np.float32)
bs        = CFG['infer_batch_size']
n_offsets = len(CFG['overlap_offsets'])

for stem, group in tqdm(sub_df.groupby('stem'), desc='Soundscapes'):
    ogg_path = os.path.join(TEST_SOUND_DIR, f'{stem}.ogg')
    if not os.path.exists(ogg_path):
        continue

    waveform, sr = torchaudio.load(ogg_path)
    if sr != CFG['sample_rate']:
        waveform = torchaudio.functional.resample(waveform, sr, CFG['sample_rate'])
    audio = waveform.mean(dim=0)  # モノラル化

    indices  = group.index.tolist()
    end_secs = group['end_sec'].tolist()

    # 重複チャンク生成: shape (N * n_offsets, n_samples)
    chunks = extract_chunks_overlap(audio, end_secs)

    # バッチ推論
    chunk_preds = np.zeros((len(chunks), len(LABELS)), dtype=np.float32)
    for i in range(0, len(chunks), bs):
        specs = audio_to_melspec(chunks[i:i + bs])  # (B, 1, n_mels, T)
        with torch.no_grad():
            logits = model(specs)
            preds  = torch.sigmoid(logits).numpy()  # CE訓練 → sigmoid推論
        chunk_preds[i:i + len(preds)] = preds

    # n_offsets 分を平均して各 end_sec の予測値を得る
    for j, idx in enumerate(indices):
        seg_preds = chunk_preds[j * n_offsets:(j + 1) * n_offsets]  # (n_offsets, num_classes)
        all_preds[idx] = seg_preds.mean(axis=0)

print('Inference done.')

In [ ]:
# ── submission.csv の保存 ──────────────────────────────────────
result_df = pd.DataFrame(all_preds, columns=LABELS)
result_df.insert(0, 'row_id', sub_df['row_id'].values)
result_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved: shape={result_df.shape}')
result_df.head(3)